In [1]:
# LangGraph에서 상태 그래프를 만들기 위한 클래스와
# 그래프의 시작/끝을 나타내는 상수를 가져옵니다.
from langgraph.graph import StateGraph, START, END

# TypedDict: 딕셔너리 형태의 타입(키/값 타입을 명시한 dict)을 정의할 수 있게 해줍니다.
# Annotated: 타입에 추가 정보를 붙일 수 있게 해줍니다. (예: 어떻게 병합할지 등)
from typing_extensions import TypedDict, Annotated

# Send: 한 노드에서 다른 노드로 "작업 요청"을 보낼 때 사용하는 타입입니다.
# 예: 같은 노드를 여러 번 호출해서 병렬/배치 처리할 때 사용합니다.
from langgraph.types import Send

# 파이썬 기본 연산 함수 모음입니다.
# 여기서는 리스트를 합칠 때 사용할 `operator.add` 를 사용합니다.
import operator

In [ ]:
# Union: 두 가지 이상의 타입(여러 타입 중 한 가지)으로 지정할 때 사용합니다.
from typing import Union

# 이 그래프에서 사용할 상태(state)의 모양을 정의합니다.
class State(TypedDict):
  # words: 단어들의 리스트. 예: ["hello", "world", ...]
  words: list[str]
  
  # output: 각 단어에 대한 정보(단어, 글자 수)를 담는 딕셔너리들의 리스트입니다.
  # - 예: [{"word": "hello", "letters": 5}, ...]
  # - Annotated[..., operator.add] 를 사용해서,
  #   같은 키(output)가 여러 번 나올 때 리스트를 이어 붙이도록(+ 연산) 지정합니다.
  output: Annotated[list[dict[str, Union[str, int]]], operator.add]

# 위에서 정의한 State 타입을 사용하는 상태 그래프를 만듭니다.
graph_builder = StateGraph(State)

In [3]:
# node_one: 전체 단어 개수를 출력만 하는 노드입니다.
# 이 노드는 상태를 변경하지 않고, 단지 "words" 리스트의 길이를 출력합니다.
def node_one(state: State):
  print(f"I want to count {len(state['words'])} words in my state.")

# node_two: 단어 하나(word)를 입력으로 받아
# 그 단어와 글자 수를 담은 딕셔너리를 output 리스트에 넣어 반환합니다.
# 이 노드는 dispatcher 에 의해 단어 개수만큼 여러 번 호출될 수 있습니다.
def node_two(word: str):
  return {
    "output": [
      {
        "word": word,
        "letters": len(word),
      }
    ]
  }

In [4]:
# 그래프에 사용할 노드들을 등록합니다.
# - "node_one": 전체 단어 개수를 출력하는 노드
# - "node_two": 단어 하나에 대한 정보를 계산하는 노드
graph_builder.add_node("node_one", node_one)
graph_builder.add_node("node_two", node_two)

# dispatcher: send API 를 이용해 node_two 를 여러 번 호출하도록 지시하는 함수입니다.
# - state["words"] 에 있는 각 단어에 대해 Send("node_two", word)를 생성합니다.
# - 즉, words 가 7개면 node_two 를 7번 호출하라는 "작업 목록"을 반환합니다.
def dispatcher(state: State):
  return [Send("node_two", word) for word in state["words"]]

# START 에서 node_one 으로 이동하는 일반 간선입니다.
graph_builder.add_edge(START, "node_one")

# add_conditional_edges 를 send API 와 함께 사용하는 패턴입니다.
# - 두 번째 인자: dispatcher 함수 → 여러 개의 Send("node_two", word) 들을 반환
# - 세 번째 인자: ["node_two"] → dispatcher 가 보내는 대상 노드 이름 목록
# 이 설정 덕분에 node_one 이후에 words 의 각 단어마다 node_two 가 실행됩니다.
graph_builder.add_conditional_edges("node_one", dispatcher, ["node_two"])

# node_two 의 처리가 끝나면 그래프를 종료합니다.
graph_builder.add_edge("node_two", END)

In [6]:
# 지금까지 정의한 노드, 상태, send API 기반 dispatcher 를 이용해서
# 실제로 실행 가능한 그래프 객체를 만듭니다.
graph = graph_builder.compile()

# graph.invoke(...) 를 호출하면 그래프를 "한 번 실행" 합니다.
# - 초기 상태로 words 리스트를 넣어줍니다.
# - 흐름:
#   1) START → node_one: 전체 단어 개수를 출력
#   2) node_one → dispatcher: words 의 각 단어에 대해 Send("node_two", word)를 생성
#   3) node_two 들이 단어 개수만큼 실행되면서
#      각 단어에 대한 {"word": ..., "letters": ...} 정보를 output 에 쌓습니다.
#   4) 모든 node_two 실행이 끝나면 END 로 이동하고,
#      최종 state 에 words 와 output 리스트가 들어 있습니다.
graph.invoke(
  {
    "words": ["hello", "world", "how", "are", "you", "doing", "today"]
  }
)

# graph

I want to count 7 words in my state.


{'words': ['hello', 'world', 'how', 'are', 'you', 'doing', 'today'],
 'output': [{'word': 'hello', 'letters': 5},
  {'word': 'world', 'letters': 5},
  {'word': 'how', 'letters': 3},
  {'word': 'are', 'letters': 3},
  {'word': 'you', 'letters': 3},
  {'word': 'doing', 'letters': 5},
  {'word': 'today', 'letters': 5}]}